# Find flood event depths

Inputs
------
- River observation points (fluvial flooding)
- Precipitation observation points (surface water flooding)
- Hazard accumulation zones
- Event return periods at observation points, for each scenario (observed, present, future RCPs)
- Return period flood maps
- Exposure points (these are user defined - here we pick a point in each cell of the return period
  flood maps which has any exposure across all return periods)

Outputs
-------
- Flood depth at each exposure point, for each event, for each scenario
- Fixed return period maps for future scenarios derived from events

In [ ]:
import os
import re
import warnings
from glob import glob

# ignore warnings about geopandas parquet implementation
warnings.filterwarnings('ignore', message='.*initial implementation of Parquet.*')
# ignore warnings about GEOS-PyGEOS conversions
warnings.filterwarnings('ignore', message='.*incompatible with the GEOS version.*')

import numpy as np
import pandas as pd
import geopandas as gpd
import pygeos.creation
import rasterio
import rioxarray
import xarray

from numba import jit
from scipy.stats.mstats import gmean
from snail.core.intersections import get_cell_indices
from tqdm.notebook import tqdm
tqdm.pandas()

In [ ]:
def latlon_to_gdf(df, lat_column='lat', lon_column='lon'):
    geometry = pygeos.creation.points(df[lon_column], df[lat_column])
    return gpd.GeoDataFrame(df, crs="EPSG:4326", geometry=geometry)

In [ ]:
# Hydrological Accumulation Zones - for river flooding
# 'T500_ID', 'T500_Type', 'T1000_ID', 'T1000_Type', 'Country', 'Area_km2', 'geometry'
hydrological_accumulation_zones = gpd.read_file("inputs/event_points_hydrological_units/JM_HAZ_T500_02.shp") \
    [['T500_ID', 'geometry']]

# Precipitation Observation Points
# 'op.id', 'op.lon', 'op.lat', 'region', 'sub.region', 'agg.zone'
precipitation_ops = latlon_to_gdf(
    pd.read_csv("inputs/event_data/PrcipOPInfo.csv"), 
    lat_column='op.lat', lon_column='op.lon') \
    [['op.id','geometry']]

# River Observation Points (OP)
# 'op.id', 'op.lon', 'op.lat', 'region', 'sub.region', 'agg.zone', 'catchment.area', 'cent.lon', 'cent.lat'
river_ops = latlon_to_gdf(
    pd.read_csv("inputs/event_data/RiverOPInfo.csv"), 
    lat_column='op.lat', lon_column='op.lon') \
    [['op.id','geometry']]

# Events - ignored
# 'event.id', 'start.day.id', 'start.year', 'start.month', 'duration', 'extent', 
# 'is.river', 'is.precip', 'track.id', ... 'JAM'
#
# Note that the series of three-letter country codes columns includes JAM
# with True/False values, but we ignore this and just include all events
# as all seem to relate to Jamaica OPs.
#
# We also ignore the is.river / is.precip columns, as the *events_rp file
# seems to disagree in terms of whether a precipitation or river OP is
# included, which leaves only the 'event.id' and date/time columns of interest
# for reporting/metadata.
#
# observed_events = pd.read_csv("inputs/event_data/ObsEventInfo.csv")
# simulated_events = pd.read_csv("inputs/event_data/SimEventInfo.csv")

# Event-OP return period
# 'event.id', 'op.id', 'rp', 'peak.day.id', 'start.day.id', 'end.day.id'
observed_events = pd.read_csv("inputs/event_data/ObsEventRP.csv", usecols=['event.id', 'op.id', 'rp'])
simulated_events = pd.read_csv("inputs/event_data/SimEventRP.csv", usecols=['event.id', 'op.id', 'rp'])

In [ ]:
len(set(simulated_events['event.id']))

In [ ]:
# simulated events - same set as in SimEventInfo.csv, conditioned for different climate scenarios
# RCP: 2.6/4.5/8.5, epoch: 2050/2080
# 'event.id', 'op.id', 'rp', 'peak.day.id', 'start.day.id', 'end.day.id'
sorted(glob("inputs/future_event_sets/SimEventRP*"))

In [ ]:
# Link River OPs to HAZs
haz_to_river_within = hydrological_accumulation_zones.sjoin(river_ops, predicate='contains', how='right') \
    [['T500_ID', 'op.id']]

In [ ]:
# some HAZ do not contain an OP
haz_without_op = hydrological_accumulation_zones[
    ~hydrological_accumulation_zones.T500_ID.isin(haz_to_river_within.T500_ID.unique())]
haz_to_river_nearest = haz_without_op.sjoin_nearest(river_ops, how='left') \
    [['T500_ID', 'op.id']]

In [ ]:
haz_to_river = pd.concat([haz_to_river_within, haz_to_river_nearest])

In [ ]:
# Read flood maps, use all cells with any depth > 0 as potential exposure points
def read_rp_map(fname):
    rp = re.search(r'Q(\d+)_', fname).group(1)
    colname = f'rp{rp}'
    with rasterio.open(fname) as dataset:
        data = dataset.read(1)
        data[data == dataset.nodata] = np.nan
        df = pd.DataFrame({colname:data.flatten()})
        
    df = df[df[colname] > 0].dropna()
    return df

def get_xy_df(fname):
    # read a raster file, convert to dataframe retaining only x,y coordinate values 
    # and 0..n index
    xy = rioxarray.open_rasterio(fname) \
        .to_dataframe('data') \
        .reset_index() \
        .drop(columns=['band','spatial_ref', 'data'])
    return xy

def read_rp_maps_to_points(pattern):
    # Define as an iter to use each file once
    rp_maps = iter(sorted(glob(pattern)))

    # Read first
    rp_points = read_rp_map(next(rp_maps))
    # Read the rest
    for fname in rp_maps:
        df = read_rp_map(fname)
        rp_points = rp_points.join(df, how='outer')
        

    # Fill NA with zeros
    rp_points = rp_points.fillna(0)
    # Include RP 2 as zero
    rp_points['rp2'] = 0
    
    # Include coordinates
    xy = get_xy_df(fname)
    rp_points = rp_points.join(xy)
    
    # name index
    rp_points.index.rename('cell_index', inplace=True)

    # Convert to GeoDataFrame
    return latlon_to_gdf(rp_points, lat_column='y', lon_column='x') \
        .drop(columns=['y', 'x'])

In [ ]:
river_rp_points = read_rp_maps_to_points('inputs/fluvial_raw_fld_depth/JM_FLRF_UD_*-aligned.tif')

# (T500_ID) cell_index, rp100, rp1500, rp200, rp20, rp500, rp50, rp2, geometry
river_exposure_points = river_rp_points.sjoin(hydrological_accumulation_zones, predicate='within', how='left') \
    .reset_index() \
    .drop(columns='index_right') \
    .set_index('T500_ID')

In [ ]:
river_exposure_points.head(2)

In [ ]:
# sense-check to confirm tiffs share a common grid
for fname in glob('inputs/fluvial_raw_fld_depth/JM_FLRF_UD_*aligned.tif'):
    with rasterio.open(fname) as ds:
        print(ds.width, ds.height)

In [ ]:
river_exposure_points.to_parquet('outputs/flrf_exposure_points.gpq')

In [ ]:
precip_rp_points = read_rp_maps_to_points('inputs/surface_water_raw_fld_depth/JM_FLSW_UD_*-aligned.tif')

In [ ]:
# (op.id) cell_index, rp100, rp1500, rp200, rp20, rp500, rp50, rp2, geometry
precip_exposure_points = precip_rp_points.sjoin_nearest(precipitation_ops, how='left') \
    .reset_index() \
    .drop(columns='index_right') \
    .set_index('op.id')

In [ ]:
precip_exposure_points.to_parquet('outputs/flsw_exposure_points.gpq')

In [ ]:
# sense-check to confirm tiffs share a common grid
for fname in glob('inputs/surface_water_raw_fld_depth/JM_FLSW_UD_*aligned.tif'):
    with rasterio.open(fname) as ds:
        print(ds.width, ds.height)

# Interpolation of flood depths at points/cells

In [ ]:
RPS = np.array([1e-3, 2, 20, 50, 100, 200, 500, 1500, 1e6])

def interpolate_rp_factor(df):
    return (
        (np.log(df.rp) - np.log(df.rp_l))
        / (np.log(df.rp_u) - np.log(df.rp_l)))

def interpolate_depth_df(df):
    depth = df.depth_l + (
        (df.depth_u - df.depth_l)
        * df.rp_factor)
        
    return depth

In [ ]:
def interpolate_event_exposure(event_zones, exposure_points, hazard_prefix):
    # Cap at max RP 1500
    event_zones.loc[event_zones.rp >= 1500, 'rp'] = 1500
    
    bin_index = np.searchsorted(RPS, event_zones.rp, side='left')
    event_zones['bin_index'] = bin_index
    event_zones['rp_l'] = RPS[bin_index - 1]
    event_zones['rp_u'] = RPS[bin_index]
    event_zones['rp_factor'] = interpolate_rp_factor(event_zones)
    # event_zones is now a dataframe with:
    # (T500_ID/op.id, event.id) rp, bin_index, rp_l, rp_u, rp_factor
    
    for e in tqdm(event_zones.reset_index()['event.id'].unique()):
        # Each HAZ in this event, with RP values
        event_haz = event_zones.loc[e]
        # All points for this event, joined with RP values via HAZ
        event_points = exposure_points.join(event_haz).dropna()
        event_points.bin_index = event_points.bin_index.astype(np.int32)

        if len(event_points):
            # Interpolate depth
            depths = [
                0, event_points.rp2, event_points.rp20, event_points.rp50, 
                event_points.rp100, event_points.rp200, event_points.rp500, event_points.rp1500
            ]
            event_points['depth_l'] = np.choose(event_points.bin_index - 1, depths)
            event_points['depth_u'] = np.choose(event_points.bin_index, depths)
            event_points['depth'] = interpolate_depth_df(event_points)
            # Any RP < 2 gets zero depth
            event_points.loc[event_points.rp <= 2, 'depth'] = 0
            
            # Output cells
            event_points = event_points[['depth', 'cell_index']]
            event_points = event_points[event_points.depth > 0]
            event_points['event'] = e
            # (T500_ID/op.id) depth, cell_index, event
            # `hazard_prefix` should include RCP, epoch metadata from event set
            event_points.to_parquet(f"outputs/{hazard_prefix}_{e}.parquet")

In [ ]:
for event_set in [observed_events]:
    # Set index and extract only return period column
    # (op.id, event.id) rp
    events = event_set.copy().set_index(['op.id', 'event.id'])
    
    # Extract precipitation events
    # (op.id, event.id) rp
    precipitation_events = events.join(precipitation_ops.set_index('op.id')) \
        .dropna() \
        .drop(columns='geometry') \
        .reset_index() \
        .set_index(['event.id', 'op.id'])
    
    # Calculate precipitation event exposure
    interpolate_event_exposure(precipitation_events, precip_exposure_points, hazard_prefix='FLSW')
    
    # Link event river OPs to HAZ (drop precipitation OPs which are not linked)
    # (op.id, event.id) rp, T500_ID
    river_events = events.reset_index().merge(haz_to_river, on='op.id').dropna()
    
    # Assert river + precip >= all (could have increased when linking river HAZ)
    assert len(river_events) + len(precipitation_events) - len(events) >= 0
    
    # Take the geometric mean of Event/OP return periods if multiple OPs per HAZ.
    # (T500_ID, event.id) rp
    river_events_haz = river_events.groupby(['event.id', 'T500_ID']).agg({'rp': gmean})
    
    # Calculate river event exposure
    interpolate_event_exposure(river_events_haz, river_exposure_points, hazard_prefix='FLRF')

## Postprocess event depths to GPKG (points) or TIFF (raster)

In [ ]:
def read_transform(fname):
    with rasterio.open(fname) as dataset:
        crs = dataset.crs
        ncols = dataset.width
        nrows = dataset.height
        transform = dataset.transform
    return crs, ncols, nrows, transform

In [ ]:
def save_to_gpkg(df, slug):
    df = df.reset_index()
    if 'T500_ID' in df.columns:
        zone_id = 'T500_ID'
    else:
        zone_id = 'op.id'
    gdf = gpd.GeoDataFrame(df[[zone_id,'depth','geometry']])
    gdf.to_file(os.path.join('outputs', f'{slug}.gpkg'), driver='GPKG')
    
def save_to_tif(df, slug, nrows, ncols, transform):
    data = np.zeros((nrows, ncols))
    
    for cell in df.itertuples():
        data[cell.row, cell.col] = cell.depth

    with rasterio.open(
            os.path.join('outputs', f'{slug}.tif'),
            'w',
            driver='GTiff',
            height=nrows,
            width=ncols,
            count=1,
            dtype=data.dtype,
            crs='+proj=latlong',
            transform=transform,
            compress='lzw'
        ) as dataset:
        dataset.write(data, 1)


In [ ]:
outputs = glob('outputs/FLSW*.parquet')
crs, ncols, nrows, transform = read_transform(glob('inputs/fluvial_raw_fld_depth/JM_FLRF_UD_*-aligned.tif')[0])

for fname in tqdm(outputs):
    df = pd.read_parquet(fname)
    slug, _ = os.path.splitext(os.path.basename(fname))
    print(slug)
    rows, cols = np.unravel_index(df.cell_index, (nrows,ncols))
    df['row'] = rows
    df['col'] = cols
    lons, lats = transform * (rows, cols)
    df['geometry'] = pygeos.creation.points(lons, lats)
    save_to_gpkg(df, slug)
    save_to_tif(df, slug, nrows, ncols, transform)

In [ ]:
from numpy.random import default_rng
rng = default_rng()

def rowcol_from_index(i, ncols):
    """i can be integer or numpy array of integers
    """
    row = i // ncols
    col = i % ncols
    return row, col

def index_from_rowcol(row, col, ncols):
    """row and col can be integers or numpy arrays of integers
    """
    return row * ncols + col

nrows = 2
ncols = 3
shape = (nrows,ncols)
data = rng.integers(low=0, high=10, size=shape)
flat = data.flatten()
reshaped = flat.reshape(shape)

for i in range(data.size):
    ncols = shape[1]
    row, col = rowcol_from_index(i, ncols)
    assert i == index_from_rowcol(row, col, ncols)
    assert reshaped[row, col] == flat[i]
    print(i, (row, col), flat[i])

print()
print("data")
print(data)
print()

# generate flat index from array size
flat_index = np.arange(nrows*ncols)
print("flat_index", flat_index)

# generate row index by repeating elements
row_indices = np.repeat(np.arange(nrows), ncols)
print("row_indices", row_indices)
# calculate row index from flat index, floor division
print(flat_index // ncols)

# generate col index by tiling range
col_indices = np.tile(np.arange(ncols), nrows)
print("col_indices", col_indices)
# calculate col index from flat index, modulo
print(flat_index % ncols)

# calculate (row_indices, col_indices) from flat_index
print(rowcol_from_index(flat_index, ncols))
# equivalent
print(np.unravel_index(flat_index, (nrows,ncols)))

# calculate flat_index from row_indices, col_indices
print(index_from_rowcol(row_indices, col_indices, ncols))
# equivalent with np.ravel_multi_index
print(np.ravel_multi_index([row_indices, col_indices], (nrows,ncols)))

In [ ]:
# best times
# 17 7 - initial
# 14 5 - no ifs in apply ~4 it/s
# ~40 it/s np.choose to get depths, then vectorized interpolation calculation
# back to ~4 it/s with Jamaica

## Don't forget
# output filenames with RCP (including baseline) and epoch

# check output size - around 30k-250k for sample, around 1M-10M for Jamaica
# hold on to cell id/index as integer - can always recover 2D index via unravel_index and geometry via transform